# PP-MAE — All 4 Options Comparison (Google Colab)

**Pathology-Prior Masked Autoencoder for Brain MRI Denoising**

| Round | PP-MAE Option | Baselines compared |
|-------|--------------|--------------------|
| 1 | CNN U-Net (Option 1) | DnCNN, UNet-L1, Noise2Noise, REDNet |
| 2 | ViT MAE 2D (Option 2) | VanillaMAE, SparK-CNN |
| 3 | Full Pipeline (Option 3) | MultiTaskUNet, TransUNet, UNETR, SwinUNETR, SeqPipeline |
| 4 | Swin Transformer (Option 4) | SwinIR-lite, Uformer-lite |

**Before running:** `Runtime → Change runtime type → GPU (T4 or better)`

## Step 1 — GPU Check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu}  ({mem:.1f} GB VRAM)')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → GPU')

print(f'PyTorch: {torch.__version__}')

## Step 2 — Install Dependencies

In [ ]:
%%capture
!pip install nibabel scikit-image matplotlib numpy torch torchvision kagglehub

## Step 3 — Clone the Repository

In [ ]:
import os, sys

REPO_DIR = '/content/AL-ML'
BRANCH   = 'claude/general-session-gviGa'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest...')
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} https://github.com/abizbright1/AL-ML.git {REPO_DIR}

sys.path.insert(0, os.path.join(REPO_DIR, 'pp_mae'))
print('Files:', os.listdir(REPO_DIR))

## Step 4 — BraTS Data

**Pick ONE option below (A, B, C, or D) — set its flag to `True`, leave the others `False`.**

| Option | Data source | Persists after session? |
|--------|-------------|------------------------|
| **A** | Kaggle download (BraTS 2023) | No — re-downloads each session (~2 min) |
| **B** | Google Drive zip (BraTS 2021) | Yes — upload once, reuse |
| **C** | Direct upload to Colab | No — re-upload each session |
| **D** | Demo / synthetic | Always — no data needed |

In [ ]:
# ============================================================
# OPTION A — Kaggle BraTS 2023  (rafi01001/brats-2023)
#
# First time only: add your Kaggle API key in Colab:
#   1. Go to kaggle.com → Your Profile → Settings → API → Create New Token
#   2. Download kaggle.json
#   3. Run the cell below to upload it, OR paste your username/key directly
# ============================================================
USE_KAGGLE = True   # ← set True to use this option

if USE_KAGGLE:
    # --- Kaggle credentials (choose one method) ---

    # Method 1: upload kaggle.json (a file picker will appear)
    # from google.colab import files
    # files.upload()   # select your kaggle.json
    # !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

    # Method 2: paste credentials directly (replace the placeholders)
    KAGGLE_USERNAME = 'your_kaggle_username'   # ← paste here
    KAGGLE_KEY      = 'your_kaggle_api_key'    # ← paste here
    import os
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
        import json
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    !chmod 600 ~/.kaggle/kaggle.json

    # Download the dataset
    import kagglehub
    print('Downloading BraTS 2023 from Kaggle (this may take a few minutes)...')
    raw_path = kagglehub.dataset_download('rafi01001/brats-2023')
    print(f'Downloaded to: {raw_path}')
    print('Contents:', os.listdir(raw_path)[:10])

    # Auto-detect the actual subject directory root
    # (kagglehub sometimes nests files 1-2 levels deep)
    sys.path.insert(0, os.path.join(REPO_DIR, 'pp_mae'))
    from brats_loader import find_brats_root, detect_brats_version

    BRATS_ROOT = find_brats_root(raw_path)
    version    = detect_brats_version(BRATS_ROOT)

    print(f'\n✅ BraTS root detected: {BRATS_ROOT}')
    print(f'   Dataset version     : BraTS {version}')
    print(f'   Subjects found      : {len([d for d in os.listdir(BRATS_ROOT) if os.path.isdir(os.path.join(BRATS_ROOT, d))])}')

    # Show a sample subject to confirm files look right
    sample_subj = sorted(os.listdir(BRATS_ROOT))[0]
    sample_path = os.path.join(BRATS_ROOT, sample_subj)
    if os.path.isdir(sample_path):
        print(f'\n   Sample subject: {sample_subj}')
        print(f'   Files: {sorted(os.listdir(sample_path))}')

In [ ]:
# ============================================================
# OPTION B — Google Drive (upload brats_datat.zip to Drive first)
# ============================================================
USE_DRIVE = False   # ← set True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ZIP = '/content/drive/MyDrive/brats_datat.zip'   # ← your Drive path

    if os.path.exists(DRIVE_ZIP):
        print(f'Found: {DRIVE_ZIP}  — extracting...')
        !mkdir -p /content/BraTS2021_data
        !unzip -q {DRIVE_ZIP} -d /content/BraTS2021_data
        !for f in /content/BraTS2021_data/*.tar; do tar -xf "$f" -C /content/BraTS2021_data/ 2>/dev/null || true; done
        BRATS_ROOT = '/content/BraTS2021_data'
        print('Done.')
    else:
        print(f'❌ Not found: {DRIVE_ZIP}')
        print('   Upload brats_datat.zip to Google Drive first.')

In [ ]:
# ============================================================
# OPTION C — Direct upload to Colab session (temporary)
# ============================================================
USE_UPLOAD = False   # ← set True

if USE_UPLOAD:
    from google.colab import files as colab_files
    print('Select your brats_datat.zip file...')
    uploaded = colab_files.upload()
    zip_name = list(uploaded.keys())[0]
    !mkdir -p /content/BraTS2021_data
    !unzip -q {zip_name} -d /content/BraTS2021_data
    !for f in /content/BraTS2021_data/*.tar; do tar -xf "$f" -C /content/BraTS2021_data/ 2>/dev/null || true; done
    BRATS_ROOT = '/content/BraTS2021_data'
    print('Done.')

In [ ]:
# ============================================================
# OPTION D — Demo / synthetic (no data needed)
# ============================================================
USE_DEMO = False   # ← set True

if USE_DEMO:
    BRATS_ROOT = None
    print('Demo mode: synthetic data will be generated automatically.')

In [ ]:
# ── Final verification — confirms BRATS_ROOT is set and has data ──────────────
import os, sys
sys.path.insert(0, os.path.join(REPO_DIR, 'pp_mae'))

if 'BRATS_ROOT' not in dir() or BRATS_ROOT is None:
    BRATS_ROOT = None
    print('⚠️  BRATS_ROOT not set — will run in demo mode.')
    print('   To use real data, set USE_KAGGLE=True (or USE_DRIVE / USE_UPLOAD) above.')
else:
    from brats_loader import detect_brats_version
    subdirs = [d for d in os.listdir(BRATS_ROOT)
               if os.path.isdir(os.path.join(BRATS_ROOT, d))]
    version = detect_brats_version(BRATS_ROOT)
    print(f'✅ BRATS_ROOT  : {BRATS_ROOT}')
    print(f'   Version     : BraTS {version}  (loader handles both 2021 and 2023 automatically)')
    print(f'   Subjects    : {len(subdirs)}')
    if subdirs:
        s = os.path.join(BRATS_ROOT, sorted(subdirs)[0])
        print(f'   Sample files: {sorted(os.listdir(s))[:6]}')

## Step 5 — Configuration

In [ ]:
# ============================================================
#  EXPERIMENT SETTINGS  — adjust before running
# ============================================================

ROUNDS       = '2,3,4'   # 1=CNN  2=ViT  3=Pipeline  4=Swin
MAX_SUBJECTS = 30        # how many BraTS subjects to load
EPOCHS       = 20        # training epochs per model
SEG_EPOCHS   = 20        # segmentor pre-training epochs
PATCH_SIZE   = 96        # spatial crop (must be divisible by 8)
SIGMA        = 0.08      # Rician noise sigma

# Save results to Drive if mounted, otherwise local
OUT_DIR = ('/content/drive/MyDrive/PP_MAE_Results'
           if os.path.exists('/content/drive/MyDrive') else
           '/content/PP_MAE_Results')
os.makedirs(OUT_DIR, exist_ok=True)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'  Rounds    : {ROUNDS}')
print(f'  Subjects  : {MAX_SUBJECTS}')
print(f'  Epochs    : {EPOCHS}')
print(f'  Device    : {DEVICE}')
print(f'  BraTS root: {BRATS_ROOT}')
print(f'  Output    : {OUT_DIR}')

## Step 6 — Run Experiments

In [ ]:
import subprocess

cmd = [
    sys.executable,
    f'{REPO_DIR}/run_all_options.py',
    '--epochs',       str(EPOCHS),
    '--seg_epochs',   str(SEG_EPOCHS),
    '--patch_size',   str(PATCH_SIZE),
    '--sigma',        str(SIGMA),
    '--max_subjects', str(MAX_SUBJECTS),
    '--rounds',       ROUNDS,
    '--out',          OUT_DIR,
]
if BRATS_ROOT:
    cmd.insert(2, BRATS_ROOT)

print('Command:', ' '.join(cmd))
print('─' * 70)

proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

print('\n✅ Done!' if proc.returncode == 0 else f'\n❌ Exit code {proc.returncode}')

## Step 7 — View Results

In [ ]:
import pandas as pd, glob
from IPython.display import display, Image

for csv in [os.path.join(OUT_DIR, 'options_results.csv'),
            os.path.join(REPO_DIR, 'options_results.csv')]:
    if os.path.exists(csv):
        df = pd.read_csv(csv)
        display(df.style.highlight_max(
            subset=df.select_dtypes('number').columns, color='#d4edda'))
        break

for f in sorted(glob.glob(os.path.join(REPO_DIR, 'options_*.png')) +
                glob.glob(os.path.join(OUT_DIR,  'options_*.png'))):
    print(f'▶ {os.path.basename(f)}')
    display(Image(f))

## Step 8 — 4-Option Head-to-Head Comparison

In [ ]:
for csv in [os.path.join(REPO_DIR, 'options_results.csv'),
            os.path.join(OUT_DIR,  'options_results.csv')]:
    if os.path.exists(csv):
        r = subprocess.run(
            [sys.executable, f'{REPO_DIR}/compare_options.py', '--csv', csv],
            capture_output=True, text=True)
        print(r.stdout)
        for f in sorted(glob.glob(os.path.join(REPO_DIR, 'options_compare_*.png'))):
            display(Image(f))
        break

## Step 9 — Grading Pipeline

In [ ]:
cmd_g = [sys.executable, f'{REPO_DIR}/run_grading.py',
         '--max_subjects', str(MAX_SUBJECTS), '--out', OUT_DIR]
if BRATS_ROOT:
    cmd_g.insert(2, BRATS_ROOT)

proc_g = subprocess.Popen(cmd_g, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc_g.stdout:
    print(line, end='', flush=True)
proc_g.wait()

for f in sorted(glob.glob(os.path.join(REPO_DIR, 'grading_*.png'))):
    display(Image(f))

## Step 10 — Download All Results

In [ ]:
import zipfile
from google.colab import files as colab_files

all_outputs = sorted(set(
    glob.glob(os.path.join(REPO_DIR, 'options_*.png')) +
    glob.glob(os.path.join(REPO_DIR, 'options_*.csv')) +
    glob.glob(os.path.join(REPO_DIR, 'grading_*.png')) +
    glob.glob(os.path.join(REPO_DIR, 'grading_*.csv')) +
    glob.glob(os.path.join(OUT_DIR,  '*.png')) +
    glob.glob(os.path.join(OUT_DIR,  '*.csv'))
))

if all_outputs:
    zip_path = '/content/PP_MAE_results.zip'
    with zipfile.ZipFile(zip_path, 'w') as zf:
        for f in all_outputs:
            zf.write(f, os.path.basename(f))
    print(f'Packed {len(all_outputs)} files → PP_MAE_results.zip')
    colab_files.download(zip_path)
else:
    print('No outputs yet — run Steps 6–9 first.')

---
## Individual Rounds (run separately for faster iteration)

In [ ]:
# Round 1 — CNN Family
cmd = [sys.executable, f'{REPO_DIR}/run_all_options.py', '--rounds', '1',
       '--epochs', str(EPOCHS), '--max_subjects', str(MAX_SUBJECTS), '--out', OUT_DIR]
if BRATS_ROOT: cmd.insert(2, BRATS_ROOT)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

In [ ]:
# Round 2 — ViT/MAE Family
cmd = [sys.executable, f'{REPO_DIR}/run_all_options.py', '--rounds', '2',
       '--epochs', str(EPOCHS), '--max_subjects', str(MAX_SUBJECTS), '--out', OUT_DIR]
if BRATS_ROOT: cmd.insert(2, BRATS_ROOT)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

In [ ]:
# Round 3 — Pipeline Family
cmd = [sys.executable, f'{REPO_DIR}/run_all_options.py', '--rounds', '3',
       '--epochs', str(EPOCHS), '--max_subjects', str(MAX_SUBJECTS), '--out', OUT_DIR]
if BRATS_ROOT: cmd.insert(2, BRATS_ROOT)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

In [ ]:
# Round 4 — Swin Family
cmd = [sys.executable, f'{REPO_DIR}/run_all_options.py', '--rounds', '4',
       '--epochs', str(EPOCHS), '--max_subjects', str(MAX_SUBJECTS), '--out', OUT_DIR]
if BRATS_ROOT: cmd.insert(2, BRATS_ROOT)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

---
## Troubleshooting

| Error | Fix |
|-------|-----|
| `403 Forbidden / host_not_allowed` on Kaggle download | Kaggle blocks datacenter IPs — use Option B or C instead |
| `Missing modalities` warning | BraTS 2023 naming auto-detected; check `detect_brats_version()` output |
| `CUDA out of memory` | Reduce `MAX_SUBJECTS` to 15 or set `PATCH_SIZE=64` |
| `ModuleNotFoundError` | Re-run Step 3 (clone/pull repo) |
| `RuntimeError: view size` on MPS | This notebook runs on CUDA — no MPS issue on Colab |
| Session disconnects mid-run | Mount Drive (Option B) — results saved to Drive as each round finishes |